# 03 — Evaluating the document processing pipeline

> **Applied Labs** · **Domain**: MM · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/05_document_processing/03_evaluate.ipynb)

**Prerequisites**:
- [02_build.ipynb](./02_build.ipynb)

**What you will learn**:
- Building an evaluation framework with exact and fuzzy field-level metrics
- Measuring per-field extraction accuracy across document types
- Assessing validation rule precision and recall
- Creating confidence calibration plots
- Calculating per-document cost and ROI vs manual processing
- Categorising failure modes and building an improvement roadmap

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0"

import os, json, re
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from enum import Enum

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — Evaluation framework

We evaluate the IDP pipeline at **three levels**:

```
┌────────────────────┐   ┌─────────────────────┐   ┌────────────────────┐
│  Field-level       │   │  Validation          │   │  System-level      │
│  accuracy          │   │  effectiveness       │   │  performance       │
├────────────────────┤   ├─────────────────────┤   ├────────────────────┤
│ • Exact match      │   │ • Rule precision     │   │ • Throughput       │
│ • Fuzzy match      │   │ • Rule recall        │   │ • Cost per doc     │
│ • Per-field scores │   │ • False positive rate│   │ • Routing accuracy │
│ • Per-type scores  │   │ • Error detection    │   │ • End-to-end time  │
└────────────────────┘   └─────────────────────┘   └────────────────────┘
```

Each metric has a clear definition and interpretation threshold.

In [ ]:
# Metric definitions and thresholds

@dataclass
class MetricDefinition:
    """Definition of an evaluation metric."""
    name: str
    description: str
    formula: str
    target: float
    unit: str

metrics: List[MetricDefinition] = [
    MetricDefinition("exact_match", "Field value matches ground truth exactly",
                     "EM = Σ(predicted == actual) / N", 0.90, "ratio"),
    MetricDefinition("fuzzy_match", "Field value within edit distance ≤ 2 of ground truth",
                     "FM = Σ(edit_dist(pred, actual) ≤ 2) / N", 0.95, "ratio"),
    MetricDefinition("field_recall", "Fraction of GT fields that were extracted",
                     "Recall = extracted_fields / total_gt_fields", 0.95, "ratio"),
    MetricDefinition("validation_precision", "Fraction of flagged issues that are real errors",
                     "Precision = true_issues / flagged_issues", 0.85, "ratio"),
    MetricDefinition("validation_recall", "Fraction of real errors caught by validator",
                     "Recall = caught_errors / total_errors", 0.90, "ratio"),
    MetricDefinition("cost_per_doc", "Average processing cost per document",
                     "Cost = api_cost + review_cost × review_rate", 0.20, "USD"),
    MetricDefinition("throughput", "Documents processed per minute",
                     "TPM = docs / minutes", 20.0, "docs/min"),
]

print("Evaluation metrics")
print("=" * 80)
print(f"{'Metric':<25s} {'Target':>8s} {'Unit':<10s} Description")
print("-" * 80)
for m in metrics:
    print(f"{m.name:<25s} {m.target:>8.2f} {m.unit:<10s} {m.description}")

print(f"\n✓ {len(metrics)} metrics defined")

## Section 2 — Field-level accuracy

We compare each extracted field to its ground truth using both **exact match**
and **fuzzy match** (Levenshtein distance ≤ 2). Results are broken down by
field name and document type.

In [ ]:
# Field-level accuracy — exact and fuzzy match

def levenshtein_distance(s1: str, s2: str) -> int:
    """Compute Levenshtein edit distance between two strings."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev_row = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = prev_row[j + 1] + 1
            deletions = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row
    return prev_row[-1]

# ── Ground truth and extractions (from 02_build) ──
ground_truths: Dict[str, Dict[str, Any]] = {
    "doc_01_invoice_acme": {
        "invoice_number": "INV-2024-001", "vendor_name": "Acme Supplies Inc.",
        "date": "2024-03-15", "subtotal": 600.0, "tax": 48.0, "total": 648.0,
    },
    "doc_02_invoice_globaltech": {
        "invoice_number": "GT-2024-0455", "vendor_name": "GlobalTech Ltd",
        "date": "2024-04-02", "subtotal": 1350.0, "tax": 0.0, "total": 1350.0,
    },
    "doc_03_invoice_freshfoods": {
        "invoice_number": "FF-7721", "vendor_name": "FreshFoods Co.",
        "date": "2024-03-28", "subtotal": 405.0, "tax": 27.0, "total": 432.0,
    },
    "doc_04_receipt_quickmart": {
        "store_name": "QuickMart Store #42", "date": "2024-03-20",
        "subtotal": 15.97, "tax": 1.28, "total": 17.25,
    },
    "doc_05_po_techcorp": {
        "po_number": "PO-2024-0088", "vendor": "ServerParts Direct",
        "date": "2024-03-25", "subtotal": 5400.0, "tax": 0.0, "total": 5400.0,
    },
}

# Mock extractions — introduce small errors for realistic eval
extractions: Dict[str, Dict[str, Any]] = {
    "doc_01_invoice_acme": {
        "invoice_number": "INV-2024-001", "vendor_name": "Acme Supplies Inc.",
        "date": "2024-03-15", "subtotal": 600.0, "tax": 48.0, "total": 648.0,
    },
    "doc_02_invoice_globaltech": {
        "invoice_number": "GT-2024-0455", "vendor_name": "GlobalTech Ltd",
        "date": "2024-04-02", "subtotal": 1350.0, "tax": 0.0, "total": 1350.0,
    },
    "doc_03_invoice_freshfoods": {
        "invoice_number": "FF-7721", "vendor_name": "FreshFoods Co",  # missing period
        "date": "2024-03-28", "subtotal": 405.0, "tax": 27.0, "total": 432.0,
    },
    "doc_04_receipt_quickmart": {
        "store_name": "QuickMart Store #42", "date": "2024-03-20",
        "subtotal": 15.97, "tax": 1.28, "total": 17.25,
    },
    "doc_05_po_techcorp": {
        "po_number": "PO-2024-0088", "vendor": "Server Parts Direct",  # extra space
        "date": "2024-03-25", "subtotal": 5400.0, "tax": 0.0, "total": 5400.0,
    },
}

# ── Compute per-field metrics ──
field_results: List[Dict[str, Any]] = []

for doc_id in ground_truths:
    gt = ground_truths[doc_id]
    ex = extractions.get(doc_id, {})
    for field_name, gt_val in gt.items():
        ex_val = ex.get(field_name)
        gt_str = str(gt_val)
        ex_str = str(ex_val) if ex_val is not None else ""

        exact = gt_str == ex_str
        dist = levenshtein_distance(gt_str, ex_str)
        fuzzy = dist <= 2

        field_results.append({
            "doc_id": doc_id, "field": field_name,
            "ground_truth": gt_str, "extracted": ex_str,
            "exact_match": exact, "fuzzy_match": fuzzy,
            "edit_distance": dist,
        })

df = pd.DataFrame(field_results)

print("Field-level accuracy results")
print("=" * 70)
print(f"\nOverall exact match : {df['exact_match'].mean():.2%}")
print(f"Overall fuzzy match : {df['fuzzy_match'].mean():.2%}")

print("\nPer-field breakdown:")
print("-" * 50)
field_summary = df.groupby("field").agg(
    exact=("exact_match", "mean"),
    fuzzy=("fuzzy_match", "mean"),
    count=("exact_match", "count"),
).round(2)
print(field_summary.to_string())

# ── Show mismatches ──
mismatches = df[~df["exact_match"]]
if len(mismatches) > 0:
    print("\nMismatches:")
    for _, row in mismatches.iterrows():
        print(f"  ✗ {row['doc_id']} / {row['field']}")
        print(f"    GT : '{row['ground_truth']}'")
        print(f"    Ex : '{row['extracted']}'  (edit dist={row['edit_distance']})")

## Section 3 — Validation effectiveness

We measure how well the validation rules detect real errors. We inject known
errors into extractions and check whether the validator catches them
(**recall**) and whether it avoids false alarms (**precision**).

In [ ]:
# Validation precision and recall — error injection test

@dataclass
class ValidationIssue:
    """A validation issue."""
    rule_id: str
    severity: str
    field: str
    message: str

def validate_invoice_fields(data: Dict[str, Any]) -> List[ValidationIssue]:
    """Validate an invoice extraction."""
    issues: List[ValidationIssue] = []
    line_items = data.get("line_items", [])

    for i, item in enumerate(line_items):
        expected = item.get("qty", 0) * item.get("unit_price", 0)
        actual = item.get("total", 0)
        if abs(expected - actual) > 0.01:
            issues.append(ValidationIssue("ARITH-001", "error",
                f"line_items[{i}].total", f"Expected {expected:.2f}, got {actual:.2f}"))

    expected_sub = sum(item.get("total", 0) for item in line_items)
    if abs(expected_sub - data.get("subtotal", 0)) > 0.01:
        issues.append(ValidationIssue("ARITH-002", "error", "subtotal",
            f"Expected {expected_sub:.2f}"))

    sub = data.get("subtotal", 0)
    tax = data.get("tax", 0)
    expected_total = sub + tax
    if abs(expected_total - data.get("total", 0)) > 0.01:
        issues.append(ValidationIssue("ARITH-003", "error", "total",
            f"Expected {expected_total:.2f}"))

    for req in ["invoice_number", "vendor_name", "date", "total"]:
        if not data.get(req):
            issues.append(ValidationIssue("REQ-001", "error", req, "Missing required field"))

    return issues

# ── Test cases: clean vs error-injected ──
test_cases: List[Dict[str, Any]] = [
    # Clean extraction
    {"name": "clean", "has_errors": False, "data": {
        "invoice_number": "INV-001", "vendor_name": "Test Corp", "date": "2024-01-01",
        "line_items": [
            {"description": "Item A", "qty": 5, "unit_price": 10.0, "total": 50.0},
            {"description": "Item B", "qty": 3, "unit_price": 20.0, "total": 60.0},
        ],
        "subtotal": 110.0, "tax": 8.80, "total": 118.80,
    }},
    # Wrong line item total
    {"name": "bad_line_total", "has_errors": True, "expected_catches": ["ARITH-001"], "data": {
        "invoice_number": "INV-002", "vendor_name": "Test Corp", "date": "2024-01-02",
        "line_items": [
            {"description": "Item A", "qty": 5, "unit_price": 10.0, "total": 55.0},  # error
            {"description": "Item B", "qty": 3, "unit_price": 20.0, "total": 60.0},
        ],
        "subtotal": 115.0, "tax": 9.20, "total": 124.20,
    }},
    # Wrong subtotal
    {"name": "bad_subtotal", "has_errors": True, "expected_catches": ["ARITH-002"], "data": {
        "invoice_number": "INV-003", "vendor_name": "Test Corp", "date": "2024-01-03",
        "line_items": [
            {"description": "Item A", "qty": 2, "unit_price": 100.0, "total": 200.0},
        ],
        "subtotal": 250.0, "tax": 20.0, "total": 270.0,  # subtotal wrong
    }},
    # Missing required field
    {"name": "missing_field", "has_errors": True, "expected_catches": ["REQ-001"], "data": {
        "invoice_number": "", "vendor_name": "Test Corp", "date": "2024-01-04",
        "line_items": [{"description": "Item A", "qty": 1, "unit_price": 50.0, "total": 50.0}],
        "subtotal": 50.0, "tax": 4.0, "total": 54.0,
    }},
    # Wrong grand total
    {"name": "bad_total", "has_errors": True, "expected_catches": ["ARITH-003"], "data": {
        "invoice_number": "INV-005", "vendor_name": "Test Corp", "date": "2024-01-05",
        "line_items": [{"description": "Item A", "qty": 4, "unit_price": 25.0, "total": 100.0}],
        "subtotal": 100.0, "tax": 8.0, "total": 115.0,  # should be 108
    }},
]

true_positives = 0
false_positives = 0
false_negatives = 0
true_negatives = 0

print("Validation effectiveness — error injection test")
print("=" * 65)
for tc in test_cases:
    issues = validate_invoice_fields(tc["data"])
    caught_ids = [i.rule_id for i in issues]

    if tc["has_errors"]:
        expected = tc.get("expected_catches", [])
        for exp_rule in expected:
            if exp_rule in caught_ids:
                true_positives += 1
            else:
                false_negatives += 1
        # Check for extra catches (false positives from error-injected data are expected)
        extra = [c for c in caught_ids if c not in expected]
        # We don't count cascading errors from injected data as FP
    else:
        if issues:
            false_positives += len(issues)
        else:
            true_negatives += 1

    icon = "✓" if (tc["has_errors"] and issues) or (not tc["has_errors"] and not issues) else "✗"
    print(f"  {icon} {tc['name']:20s}  errors={tc['has_errors']}  caught={len(issues)}  rules={caught_ids}")

precision = true_positives / max(true_positives + false_positives, 1)
recall = true_positives / max(true_positives + false_negatives, 1)
f1 = 2 * precision * recall / max(precision + recall, 0.001)

print(f"\nValidation metrics:")
print(f"  True positives  : {true_positives}")
print(f"  False positives : {false_positives}")
print(f"  False negatives : {false_negatives}")
print(f"  Precision       : {precision:.2%}")
print(f"  Recall          : {recall:.2%}")
print(f"  F1 Score        : {f1:.2%}")

## Section 4 — Confidence calibration

A well-calibrated model should have actual accuracy proportional to its stated
confidence. We plot a **calibration curve**: binned confidence vs actual accuracy.

In [ ]:
# Confidence calibration plot

# Simulate 100 document extractions with confidence scores and accuracy
np.random.seed(42)
n_docs = 100
confidences = np.clip(np.random.beta(8, 2, size=n_docs), 0.3, 1.0)

# Actual accuracy is correlated with confidence but with noise
# Higher confidence → higher probability of correct extraction
actual_correct = np.random.random(n_docs) < (confidences * 0.95 + 0.02)

# ── Bin into 10 buckets ──
n_bins = 10
bin_edges = np.linspace(0.3, 1.0, n_bins + 1)
bin_centers: List[float] = []
bin_accuracies: List[float] = []
bin_counts: List[int] = []

for i in range(n_bins):
    mask = (confidences >= bin_edges[i]) & (confidences < bin_edges[i + 1])
    if mask.sum() > 0:
        bin_centers.append((bin_edges[i] + bin_edges[i + 1]) / 2)
        bin_accuracies.append(actual_correct[mask].mean())
        bin_counts.append(int(mask.sum()))

# ── Expected Calibration Error (ECE) ──
ece = sum(
    count * abs(acc - center)
    for center, acc, count in zip(bin_centers, bin_accuracies, bin_counts)
) / n_docs

print("Confidence calibration analysis")
print("=" * 55)
print(f"{'Bin Center':>12s} {'Accuracy':>10s} {'Count':>8s} {'Gap':>8s}")
print("-" * 42)
for c, a, n in zip(bin_centers, bin_accuracies, bin_counts):
    gap = abs(a - c)
    print(f"{c:>12.2f} {a:>10.2%} {n:>8d} {gap:>8.3f}")
print(f"\nExpected Calibration Error (ECE): {ece:.4f}")

# ── Calibration plot ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Calibration curve
ax1.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
ax1.bar(bin_centers, bin_accuracies, width=0.06, alpha=0.7, color="steelblue", label="Actual accuracy")
ax1.set_xlabel("Mean predicted confidence")
ax1.set_ylabel("Actual accuracy")
ax1.set_title(f"Calibration curve (ECE = {ece:.4f})")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Confidence histogram
ax2.hist(confidences, bins=20, alpha=0.7, color="steelblue", edgecolor="white")
ax2.axvline(x=0.95, color="red", linestyle="--", alpha=0.7, label="Auto-approve threshold")
ax2.axvline(x=0.70, color="orange", linestyle="--", alpha=0.7, label="Review threshold")
ax2.set_xlabel("Confidence score")
ax2.set_ylabel("Count")
ax2.set_title("Confidence distribution (100 documents)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 5 — Cost analysis

We calculate per-document cost across three processing approaches and model
the ROI at different volumes.

In [ ]:
# Cost analysis — per-document and annual ROI

@dataclass
class CostModel:
    """Cost model for document processing."""
    approach: str
    api_cost: float       # per document
    review_cost: float    # per document requiring review
    review_rate: float    # fraction of docs needing review
    setup_cost: float     # one-time setup
    maintenance: float    # annual maintenance

    @property
    def per_doc_cost(self) -> float:
        return self.api_cost + self.review_cost * self.review_rate

    def annual_cost(self, volume: int) -> float:
        return self.per_doc_cost * volume + self.maintenance

models: List[CostModel] = [
    CostModel("Manual entry", 0.0, 3.50, 1.0, 0, 0),
    CostModel("Template OCR", 0.02, 1.50, 0.25, 25_000, 50_000),
    CostModel("VLM pipeline", 0.05, 0.50, 0.10, 10_000, 12_000),
]

volumes = [1_000, 10_000, 50_000, 100_000, 500_000]

print("Per-document cost comparison")
print("=" * 60)
for m in models:
    print(f"  {m.approach:20s}: ${m.per_doc_cost:.2f}/doc")

print("\nAnnual cost by volume")
print("=" * 80)
header = f"{'Volume':>10s}"
for m in models:
    header += f" | {m.approach:>15s}"
print(header)
print("-" * 80)
for vol in volumes:
    row = f"{vol:>10,d}"
    for m in models:
        cost = m.annual_cost(vol)
        row += f" | ${cost:>13,.0f}"
    print(row)

# ── ROI vs manual ──
print("\nROI vs manual entry")
print("=" * 60)
manual = models[0]
for m in models[1:]:
    for vol in [10_000, 100_000]:
        manual_cost = manual.annual_cost(vol)
        ai_cost = m.annual_cost(vol) + m.setup_cost  # include first-year setup
        savings = manual_cost - ai_cost
        roi = savings / ai_cost * 100 if ai_cost > 0 else float("inf")
        print(f"  {m.approach} at {vol:,d} docs: savings=${savings:,.0f}, ROI={roi:.0f}%")

# ── Breakeven chart ──
fig, ax = plt.subplots(figsize=(10, 5))
for m in models:
    costs = [m.annual_cost(v) for v in volumes]
    ax.plot(volumes, costs, "o-", linewidth=2, label=m.approach)

ax.set_xlabel("Annual document volume")
ax.set_ylabel("Annual cost ($)")
ax.set_title("Cost comparison by processing approach")
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 6 — Failure analysis and improvement roadmap

We categorise extraction failures into an **error taxonomy** and prioritise
improvements by frequency and impact.

In [ ]:
# Failure analysis — error taxonomy and improvement roadmap

@dataclass
class FailureMode:
    """A category of extraction failure."""
    category: str
    description: str
    frequency: str    # high | medium | low
    impact: str       # high | medium | low
    example: str
    mitigation: str

failure_modes: List[FailureMode] = [
    FailureMode(
        "OCR noise", "VLM misreads characters (O vs 0, l vs 1)",
        "medium", "medium",
        "'$1,5OO' read as '$1,500' or '$1,500.OO'",
        "Pre-process with dedicated OCR; add character-level validation"
    ),
    FailureMode(
        "Layout confusion", "Fields from adjacent columns/rows are swapped",
        "medium", "high",
        "Ship-to address extracted as bill-to address",
        "Improve layout analysis; add region-aware prompting"
    ),
    FailureMode(
        "Schema mismatch", "Document has fields not in the extraction schema",
        "low", "medium",
        "Invoice with discount field not captured",
        "Dynamic schema detection; add catch-all 'other_fields' output"
    ),
    FailureMode(
        "Multi-page errors", "Fields split across pages are partially extracted",
        "high", "high",
        "Line items on page 2 are missed entirely",
        "Multi-page concatenation before extraction; page-aware prompting"
    ),
    FailureMode(
        "Currency/format", "Non-standard currency or number formats",
        "medium", "medium",
        "'EUR 1.500,00' parsed as $1500.00 instead of €1500.00",
        "Locale-aware parsing; extract currency symbol as separate field"
    ),
    FailureMode(
        "Low-quality scan", "Blurry, skewed, or faded documents",
        "high", "high",
        "Entire invoice too blurry to read vendor name",
        "Image pre-processing (deskew, sharpen, contrast); confidence thresholding"
    ),
]

print("Failure analysis — error taxonomy")
print("=" * 75)
for i, fm in enumerate(failure_modes, 1):
    priority = "🔴" if fm.frequency == "high" and fm.impact == "high" else \
               "🟡" if fm.frequency == "high" or fm.impact == "high" else "🟢"
    print(f"\n{priority} {i}. {fm.category}")
    print(f"   Description : {fm.description}")
    print(f"   Frequency   : {fm.frequency}  |  Impact: {fm.impact}")
    print(f"   Example     : {fm.example}")
    print(f"   Mitigation  : {fm.mitigation}")

# ── Improvement roadmap ──
@dataclass
class RoadmapItem:
    """A prioritised improvement task."""
    priority: int
    title: str
    effort: str
    expected_improvement: str

roadmap: List[RoadmapItem] = [
    RoadmapItem(1, "Multi-page document support", "2 weeks",
                "+15% recall on multi-page invoices"),
    RoadmapItem(2, "Image pre-processing pipeline (deskew, enhance)", "1 week",
                "+10% accuracy on low-quality scans"),
    RoadmapItem(3, "Layout-aware region prompting", "1 week",
                "-50% layout confusion errors"),
    RoadmapItem(4, "Dynamic schema detection", "2 weeks",
                "+5% field recall on diverse formats"),
    RoadmapItem(5, "Locale-aware number/currency parsing", "3 days",
                "-80% currency format errors"),
    RoadmapItem(6, "Confidence calibration fine-tuning", "1 week",
                "ECE improvement from 0.04 → 0.02"),
]

print("\n\nImprovement roadmap")
print("=" * 75)
print(f"{'P':>3s} {'Title':<45s} {'Effort':<12s} {'Expected Impact'}")
print("-" * 75)
for item in roadmap:
    print(f"{item.priority:>3d} {item.title:<45s} {item.effort:<12s} {item.expected_improvement}")

print(f"\n✓ {len(failure_modes)} failure modes identified")
print(f"✓ {len(roadmap)} improvement items prioritised")

## Exercises

1. **Extended field accuracy** — Add line-item-level evaluation: compare each
   extracted line item's description, quantity, and total against ground truth.
   Calculate per-line-item accuracy and show which items are hardest.

2. **Confidence threshold tuning** — Sweep the auto-approve threshold from 0.60
   to 0.99 and plot the tradeoff between auto-approval rate and error rate.
   Find the optimal threshold that keeps errors below 2%.

3. **Multi-approach comparison** — Simulate running the same 100 documents through
   all three cost models (manual, template OCR, VLM). Generate a comparison table
   showing accuracy, cost, and time for each approach.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Field-level eval needs both exact match and fuzzy match metrics |
| 2 | Validation precision/recall measures how well rules catch real errors |
| 3 | Confidence calibration ensures stated confidence matches actual accuracy |
| 4 | VLM pipelines break even vs manual at ~5K docs/year; vs template at ~50K |
| 5 | Multi-page documents and low-quality scans are the top failure modes |
| 6 | A prioritised roadmap converts failure analysis into actionable improvements |

## What's Next

You now have a complete IDP pipeline — from first principles through evaluation.
Next steps:
- Try **Lab 06 — AI Data Analyst** for another multimodal application
- Explore the **multimodal/** foundations for deeper VLM understanding
- Scale this pipeline with batch processing and async workers

---

## References

1. Guo, C. et al., *On Calibration of Modern Neural Networks*, ICML 2017.
2. Grand View Research, *Intelligent Document Processing Market Report*, 2024.
3. Reducto, *IDP Accuracy Benchmarks* — <https://reducto.ai/benchmarks>
4. Naeini, M. P. et al., *Obtaining Well Calibrated Probabilities Using Bayesian Binning into Quantiles*, AAAI 2015.